# Semantic Caching

**Tags:** optimization, caching
**Prerequisites:** Embeddings
**Related Concepts:** See flowchart below
**Source:** llm/concepts/semantic-caching.md

## TL;DR

Cache LLM responses by semantic similarity, not exact string match. Incoming query similar to cached query (embedding distance < threshold)? Return cached response. Reduces API costs 30-70%; typical hit rates: 30-50%. Trade-off: +5-20ms latency for embedding/similarity search, staleness risk.

## Core Intuition

Key-value caches fail on "What is the capital of France?" vs "What's France's capital?" (different strings, same meaning). Semantic caching embeds both queries, computes similarity, recognizes they're the same. This mirrors how humans cache: we don't remember exact phrasing, we remember meaning.

## How It Works

**Traditional Key-Value Caching (Wrong for LLMs):**
```
Query: "What is the capital of France?"
Cache lookup: exact_hash(query) == "abc123"?
  → No match (different phrasing)
  → Call LLM (wasted cost)

Query: "What's France's capital?"
Cache lookup: exact_hash(query) == "abc123"?
  → No match again
  → Call LLM again (wasted cost)
```

**Semantic Caching (Better):**
```
Query: "What is the capital of France?"
  ↓
1. Embed query: embed_model.encode(query) → v_q ∈ ℝ^384
2. Search cache: find similar vectors using ANN (FAISS, HNSW)
   similarity(v_q, cached_vectors) > threshold (e.g., > 0.95)?
3a. HIT: Return cached("Paris is the capital of France")
3b. MISS: Call LLM, cache response with embedding

Query: "What's France's capital?"
  ↓
1. Embed query: embed_model.encode(query) → v_q' ∈ ℝ^384
2. similarity(v_q', v_q) ≈ 0.98 > 0.95
   → HIT: Return cached response (cost saved!)
```

**Architecture:**

```
Incoming Request
    ↓
[Embedding Module: encode query]
    ↓
Query Embedding (e.g., 384-dim vector)
    ↓
[Vector Similarity Search: ANN index over cached embeddings]
    ↓
    ├─ Similarity > threshold (e.g., > 0.95)
    │  └─ CACHE HIT → Return cached response
    │
    └─ Similarity ≤ threshold
       └─ CACHE MISS → Call LLM
           ↓
           [Cache new response with embedding]
           ↓
           Return response
```

**Key Components:**

**1. Embedding Model:**
- Must match production embeddings
- Typical: sentence-transformers (BERT-based, 384-768 dims)
- Examples: "all-MiniLM-L6-v2" (384d, fast), "all-mpnet-base-v2" (768d, accurate)
- Latency: 2-5ms per query
- Cost: one-time inference (can be local/hosted)

**2. Vector Store (ANN Index):**
- FAISS: fast, no external infra, in-memory
- Pinecone/Weaviate: managed, higher latency but simpler
- HNSW (Hierarchical Navigable Small World): fast, memory-efficient
- Typical: query 1M vectors in < 5ms

**3. Similarity Threshold:**
- Too high (0.98+): rarely match, low cache hit rate
- Too low (0.85): risky, dissimilar queries return wrong answers
- Optimal: 0.90-0.95 (depends on domain)
- Cost of mismatch: return wrong cached answer (reputation risk)

**4. Cache Invalidation:**
- Time-based: expire after N hours (simple, staleness)
- Explicit: invalidate on data updates (manual, complex)
- Hybrid: assume 24-hour freshness for most queries

### Workflow Diagram

```mermaid
graph LR
    A["Input"] --> B["Semantic Caching Process"]
    B --> C["Output"]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#e8f5e9
```

**Note:** This is a basic workflow template. Review and customize based on specific concept.

## Key Properties & Trade-offs

/ Trade-offs

| Aspect | Semantic Cache | Standard Cache | No Cache |
|--------|----------------|----------------|----------|
| Hit rate | 30-50% | 0% (no similar queries) | 0% |
| Cost savings | 3-10x | 0x | 0x |
| Latency | Base + 5-20ms (embedding) | Base | Base |
| Storage | Query embeddings + responses | Responses only | Nothing |
| Staleness | hours/days | instant | instant |
| Accuracy | 99.5% (if threshold tuned) | 100% | 100% |

**Cache Hit Rate Estimates:**

| Scenario | Hit Rate | Reasoning |
|----------|----------|-----------|
| FAQ chatbot | 40-60% | Users ask similar questions frequently |
| Code documentation | 30-50% | Similar doc queries repeated |
| Product support | 35-55% | Stock questions reworded differently |
| General Q&A | 20-35% | More diverse queries |
| Novel research | 5-15% | Unique questions unlikely to repeat |

**Cost Example (OpenAI GPT-4):**
```
Scenario: 1M daily queries, $0.03 per request
Base cost: 1M × $0.03 = $30,000/day

With 40% cache hit rate:
- 600k cache hits (no cost)
- 400k LLM calls: 400k × $0.03 = $12,000/day
- Cache overhead (embeddings): ~$500/day
- Total: $12,500/day (58% cost reduction)

ROI: Saves $17,500/day
```

### Code Implementation

```python
# Key imports for Semantic Caching
import numpy as np
import torch
from typing import Any

# Semantic Caching example implementation
class SemanticCaching:
    """
    Semantic Caching implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts

```mermaid
graph TD
    A["Semantic Caching"]
    B["Embeddings"] -->|prerequisite| A
    A -->|used with| D["Vector Databases"]
    
    style A fill:#fff3e0
```

### Common Interview Questions

**Q: What is Semantic Caching used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Semantic Caching?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Semantic Caching compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Semantic Caching?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Semantic Caching?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/semantic-caching.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]